<a href="https://colab.research.google.com/github/Alisha-bhatti/DEEP-LEARNING-FOR-NATURAL-LANGUAGE-PROCESSING/blob/main/BiLSTM_NER_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q tensorflow tensorflow-datasets seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from tensorflow.keras import layers
from seqeval.metrics import classification_report

In [ ]:
dataset, info = tfds.load("conll2003", with_info=True)
train_data = dataset["train"]
val_data = dataset["dev"]
test_data = dataset["test"]
tag_vocab = info.features["ner"].feature.names
print(tag_vocab)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

...

Generating dev examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/conll2003/conll2003/incomplete.MLT83O_1.0.0/conll2003-dev.tfrecord*...:   …

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/conll2003/conll2003/incomplete.MLT83O_1.0.0/conll2003-test.tfrecord*...:  …

Dataset conll2003 downloaded and prepared to /root/tensorflow_datasets/conll2003/conll2003/1.0.0. Subsequent calls will reuse this data.
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [ ]:
def simplify_tags(tags): #Function converts fine-grained tags into 4-class system.
    new_tags = []
    for tag in tags:
        tag_name = tag_vocab[tag] #Converts numeric tag → string label.
        if "PER" in tag_name:     #Detects Person entity.
            new_tags.append(1)
        elif "LOC" in tag_name:
            new_tags.append(2)
        elif "ORG" in tag_name:
            new_tags.append(3)
        else:
            new_tags.append(0)
    return new_tags
def extract_data(data):
    sentences = []
    labels = []

    for example in data:
        words = example["tokens"].numpy().astype(str).tolist() # Convert numpy array to list of strings
        tags = simplify_tags(example["ner"].numpy())
        sentences.append(words)
        labels.append(tags)

    return sentences, labels

In [ ]:
train_sentences, train_labels = extract_data(train_data)
val_sentences, val_labels = extract_data(val_data)
test_sentences, test_labels = extract_data(test_data)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
tokenizer = Tokenizer(oov_token="<UNK>")
tokenizer.fit_on_texts(train_sentences)
X_train = tokenizer.texts_to_sequences(train_sentences)
X_val = tokenizer.texts_to_sequences(val_sentences)
X_test = tokenizer.texts_to_sequences(test_sentences)
max_len = 50

In [ ]:
X_train = pad_sequences(X_train, maxlen=max_len, padding="post")
X_val = pad_sequences(X_val, maxlen=max_len, padding="post")
X_test = pad_sequences(X_test, maxlen=max_len, padding="post")

In [ ]:
y_train = pad_sequences(train_labels, maxlen=max_len, padding="post")
y_val = pad_sequences(val_labels, maxlen=max_len, padding="post")
y_test = pad_sequences(test_labels, maxlen=max_len, padding="post")
vocab_size = len(tokenizer.word_index) + 1
model = tf.keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len),
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.TimeDistributed(layers.Dense(4, activation="softmax"))
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32
)
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 56s 111ms/step - accuracy: 0.9683 - loss: 0.1044 - val_accuracy: 0.9845 - val_loss: 0.0528
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 46s 106ms/step - accuracy: 0.9929 - loss: 0.0243 - val_accuracy: 0.9884 - val_loss: 0.0374
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 46s 104ms/step - accuracy: 0.9968 - loss: 0.0109 - val_accuracy: 0.9890 - val_loss: 0.0353
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 45s 102ms/step - accuracy: 0.9982 - loss: 0.0061 - val_accuracy: 0.9886 - val_loss: 0.0391
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 83s 104ms/step - accuracy: 0.9989 - loss: 0.0038 - val_accuracy: 0.9891 - val_loss: 0.0424
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9847 - loss: 0.0585
Test Accuracy: 0.9846958518028259


In [ ]:
y_pred = model.predict(X_test)
y_pred = np.argmax(y_pred, axis=-1)
label_map = {0:"O", 1:"PER", 2:"LOC", 3:"ORG"}

true_labels = []
pred_labels = []

for i in range(len(y_test)):
    temp_true = []
    temp_pred = []
    for j in range(max_len):
        if y_test[i][j] != 0:
            temp_true.append(label_map[y_test[i][j]])
            temp_pred.append(label_map[y_pred[i][j]])
    true_labels.append(temp_true)
    pred_labels.append(temp_pred)

print(classification_report(true_labels, pred_labels))

108/108 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ORG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LOC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


              precision    recall  f1-score   support

          ER       0.69      0.59      0.63      1168
          OC       0.79      0.74      0.77      1357

   micro avg       0.75      0.67      0.71      2525
   macro avg       0.74      0.66      0.70      2525
weighted avg       0.75      0.67      0.71      2525

